<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/LivePortrait_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<a href="https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/LivePortrait_Colab.ipynb" target="_parent"><img src="https://img.shields.io/badge/GitHub-View_Source-181717?logo=github" alt="View on GitHub"/></a>

# 🎭 LivePortrait — Fast Portrait Animation, Expression Retargeting & Video Reenactment

**Model:** [`KwaiVGI/LivePortrait`](https://huggingface.co/KwaiVGI/LivePortrait) &nbsp;|&nbsp; **GitHub:** [`KwaiVGI/LivePortrait`](https://github.com/KwaiVGI/LivePortrait) &nbsp;|&nbsp; **License:** MIT

**LivePortrait** (by Kuaishou Technology, UCAS & USTC) is a high-speed implicit-keypoint portrait animation and facial retargeting framework with **stitching and eye/lip retargeting modules**. It runs in **< 4 GB VRAM** on any Colab GPU (`T4`, `L4`, or `A100`) and serves both as a standalone portrait animator and as the ideal **Avatar Portrait Pre-Processor** for `InfiniteTalk` and `LongCat-Video-Avatar-1.5`.

### ✨ Key Features of This Colab Notebook
1. **🖼️ Image-to-Video Portrait Animation (`I2V`)**:
   - Animate any static portrait (such as Nikita or custom character portraits) using a **driving facial video (`.mp4`)**, **driving expression image (`.jpg`/`.png`)**, or **built-in motion templates (`.pkl`)** (`talking`, `laugh`, `wink`, `shake_face`, `shy`, `open_lip`).
   - Full control over **Animation Region** (`all`, `exp`, `pose`, `lip`, `eyes`), **Driving Mode** (`expression-friendly` vs `pose-friendly`), **Driving Multiplier**, **Stitching**, **Relative Motion**, and **Initial Lip Normalization**.
2. **🎞️ Video-to-Video Facial Reenactment & Lip Silencing (`V2V`)**:
   - Transfer facial expressions and head pose from a driving video onto an existing portrait video (`.mp4`) with Kalman filter temporal smoothing (`driving_smooth_observation_variance`) and automatic audio preservation.
   - Includes **Video Lip-Retargeting / Lip-Silencing** to generate natural closed-mouth idle avatar loops from speaking videos.
3. **🎛️ Interactive Portrait Expression Editor (Pitch / Yaw / Roll / Smile / Wink / Eyebrow / Mouth / Gaze)**:
   - Adjust head pose (`pitch`, `yaw`, `roll`), gently close an open mouth (`lip_ratio` / `lip_open`), tweak smile, wink, eyebrow height, or eye gaze on a static portrait, and export the full-resolution paste-back `.png` directly to Google Drive.

| GPU Tier | VRAM | Peak VRAM (FP16) | Speed (per frame) | Supported Workflows |
| :--- | :--- | :--- | :--- | :--- |
| **Colab T4** | 16 GB | **~3.2 GB** | `~25 ms` (40 FPS) | I2V Animation, V2V Reenactment, Interactive Expression Editor |
| **Colab L4** | 24 GB | **~3.2 GB** | `~14 ms` (70 FPS) | I2V Animation, V2V Reenactment, Interactive Expression Editor |
| **Colab A100** | 40 / 80 GB | **~3.2 GB** | `~9 ms` (110 FPS) | I2V Animation, V2V Reenactment, Interactive Expression Editor |

In [ ]:
# @title STEP 1 — Clone LivePortrait, Install Dependencies & Apply Colab ONNX Patch
# @markdown Clones `KwaiVGI/LivePortrait`, installs `onnxruntime-gpu` (with automatic CPU fallback), `tyro`,
# @markdown `pykalman`, `lmdb`, `albumentations`, `ffmpeg-python`, `tqdm==4.66.5`, and `gradio==5.49.1`.

import os, sys, subprocess, time, shutil, torch

if not torch.cuda.is_available():
    raise SystemExit('No CUDA GPU detected. Go to Runtime -> Change runtime type -> select T4, L4, or A100 GPU.')

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f'[GPU] {gpu_name} — {vram_gb:.1f} GB VRAM | PyTorch {torch.__version__} (CUDA {torch.version.cuda})')

LIVEPORTRAIT_REPO_DIR = '/content/LivePortrait'
if not os.path.exists(os.path.join(LIVEPORTRAIT_REPO_DIR, 'src', 'gradio_pipeline.py')):
    print(f'[Git] Cloning https://github.com/KwaiVGI/LivePortrait.git -> {LIVEPORTRAIT_REPO_DIR} ...')
    if os.path.exists(LIVEPORTRAIT_REPO_DIR):
        shutil.rmtree(LIVEPORTRAIT_REPO_DIR)
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/KwaiVGI/LivePortrait.git',
        LIVEPORTRAIT_REPO_DIR,
    ], check=True)
else:
    print(f'[Git] LivePortrait repository already present at {LIVEPORTRAIT_REPO_DIR}')

if LIVEPORTRAIT_REPO_DIR not in sys.path:
    sys.path.insert(0, LIVEPORTRAIT_REPO_DIR)

t0 = time.time()
print('[pip] Installing LivePortrait packages + onnxruntime + gradio==5.49.1 ...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'tyro>=0.8.5',
    'pykalman>=0.9.7',
    'lmdb>=1.4.1',
    'albumentations>=1.4.10',
    'scikit-image>=0.24.0',
    'ffmpeg-python>=0.2.0',
    'imageio>=2.34.0',
    'imageio-ffmpeg>=0.5.1',
    'opencv-python-headless',
    'onnx>=1.16.0',
    'rich',
    'pyyaml',
    'scipy',
    'huggingface-hub>=0.27.0',
    'tqdm==4.66.5',
    'gradio==5.49.1',
], check=True)

ort_gpu = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnxruntime-gpu'], check=False)
if ort_gpu.returncode != 0:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnxruntime>=1.18.0'], check=True)

# Patch human_landmark_runner.py & cropper.py so ONNX Runtime includes CPUExecutionProvider as safe fallback
hlr_path = os.path.join(LIVEPORTRAIT_REPO_DIR, 'src', 'utils', 'human_landmark_runner.py')
if os.path.exists(hlr_path):
    hlr_txt = open(hlr_path, 'r', encoding='utf-8').read()
    old_prov = "providers=[\n                    ('CUDAExecutionProvider', {'device_id': device_id})\n                ]"
    new_prov = "providers=[\n                    ('CUDAExecutionProvider', {'device_id': device_id}),\n                    'CPUExecutionProvider'\n                ]"
    if old_prov in hlr_txt:
        open(hlr_path, 'w', encoding='utf-8').write(hlr_txt.replace(old_prov, new_prov))

cropper_path = os.path.join(LIVEPORTRAIT_REPO_DIR, 'src', 'utils', 'cropper.py')
if os.path.exists(cropper_path):
    c_txt = open(cropper_path, 'r', encoding='utf-8').read()
    c_txt = c_txt.replace('["CUDAExecutionProvider"]', '["CUDAExecutionProvider", "CPUExecutionProvider"]')
    open(cropper_path, 'w', encoding='utf-8').write(c_txt)

print(f'[Install] Completed in {time.time() - t0:.1f}s (with ONNX provider fallback patch applied).')

In [ ]:
# @title STEP 2 — Mount Google Drive & Cache `KwaiVGI/LivePortrait` Pretrained Weights
# @markdown Mounts Google Drive, downloads `KwaiVGI/LivePortrait` weights (`liveportrait/*` and `insightface/*`)
# @markdown into `/content/drive/MyDrive/AEI_3D_Cache/LivePortrait/pretrained_weights`, and links them into `/content/LivePortrait/pretrained_weights`.

import os, sys, time, pathlib, shutil
from huggingface_hub import snapshot_download
from tqdm.notebook import tqdm

HF_REPO_ID = 'KwaiVGI/LivePortrait'
DOWNLOAD_ANIMAL_WEIGHTS = False  # @param {type:"boolean"}

try:
    from google.colab import drive
    if not pathlib.Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive', force_remount=False)
    CACHE_ROOT = pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/LivePortrait')
    HAS_DRIVE = True
except Exception as exc:
    print(f'[Cache] Google Drive unavailable ({exc}); using /content/hf_cache/LivePortrait.')
    CACHE_ROOT = pathlib.Path('/content/hf_cache/LivePortrait')
    HAS_DRIVE = False

HF_HOME_DIR = CACHE_ROOT / 'hf_home'
WEIGHTS_CACHE_DIR = CACHE_ROOT / 'pretrained_weights'
HF_HOME_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_HOME_DIR)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(HF_HOME_DIR / 'hub')
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

OUT_DIR = pathlib.Path('/content/liveportrait_out')
OUT_DIR.mkdir(parents=True, exist_ok=True)
if HAS_DRIVE:
    DRIVE_OUT = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/LivePortrait')
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
else:
    DRIVE_OUT = OUT_DIR

ignore_list = ['*.git*', 'README.md', 'docs/*']
if not DOWNLOAD_ANIMAL_WEIGHTS:
    ignore_list.append('liveportrait_animals/*')

req_file = WEIGHTS_CACHE_DIR / 'liveportrait' / 'base_models' / 'spade_generator.pth'
if not req_file.exists() or req_file.stat().st_size < 10_000_000:
    print(f'[Cache] Downloading KwaiVGI/LivePortrait weights to {WEIGHTS_CACHE_DIR} ...')
    t0 = time.time()
    snapshot_download(
        repo_id=HF_REPO_ID,
        local_dir=str(WEIGHTS_CACHE_DIR),
        local_dir_use_symlinks=False,
        ignore_patterns=ignore_list,
        tqdm_class=tqdm,
    )
    print(f'[Cache] Weights downloaded in {time.time() - t0:.1f}s.')
else:
    print(f'[Cache] LivePortrait weights already cached at {WEIGHTS_CACHE_DIR}')

repo_weights_dir = pathlib.Path(LIVEPORTRAIT_REPO_DIR) / 'pretrained_weights'
if repo_weights_dir.is_symlink() or repo_weights_dir.exists():
    if repo_weights_dir.is_symlink():
        repo_weights_dir.unlink()
    elif repo_weights_dir.is_dir() and str(repo_weights_dir.resolve()) != str(WEIGHTS_CACHE_DIR.resolve()):
        shutil.rmtree(repo_weights_dir)
if not repo_weights_dir.exists():
    os.symlink(str(WEIGHTS_CACHE_DIR), str(repo_weights_dir))

print(f'[Setup] Symlinked {repo_weights_dir} -> {WEIGHTS_CACHE_DIR}')
print(f'[Setup] Output directory: {DRIVE_OUT}')

In [ ]:
# @title STEP 3 — LivePortrait Engine (I2V Animation, V2V Reenactment & Interactive Expression Editor)
# @markdown Initializes `LivePortraitEngine` wrapping upstream `GradioPipeline` (`F`, `M`, `W`, `G`, `S` modules
# @markdown + `InsightFace` 106-point detector + 203-point `LandmarkRunner`) for I2V, V2V, and single-image expression editing.

import os, sys, time, uuid, shutil, pathlib, traceback
from typing import Dict, List, Optional, Tuple
import cv2, torch, numpy as np
from PIL import Image
from IPython.display import display, FileLink

if LIVEPORTRAIT_REPO_DIR not in sys.path:
    sys.path.insert(0, LIVEPORTRAIT_REPO_DIR)

from src.config.argument_config import ArgumentConfig
from src.config.inference_config import InferenceConfig
from src.config.crop_config import CropConfig
from src.gradio_pipeline import GradioPipeline, update_args

EXAMPLES_DIR = pathlib.Path(LIVEPORTRAIT_REPO_DIR) / 'assets' / 'examples'
SOURCE_EXAMPLES_DIR = EXAMPLES_DIR / 'source'
DRIVING_EXAMPLES_DIR = EXAMPLES_DIR / 'driving'

def list_builtin_examples() -> Tuple[Dict[str, str], Dict[str, str], Dict[str, str]]:
    src_imgs, drv_vids, drv_pkls = {}, {}, {}
    if SOURCE_EXAMPLES_DIR.exists():
        for p in sorted(SOURCE_EXAMPLES_DIR.glob('*.jpg')):
            src_imgs[f'Portrait {p.name}'] = str(p)
    if DRIVING_EXAMPLES_DIR.exists():
        for p in sorted(DRIVING_EXAMPLES_DIR.glob('*.mp4')):
            drv_vids[f'Driving Video {p.name}'] = str(p)
        for p in sorted(DRIVING_EXAMPLES_DIR.glob('*.pkl')):
            drv_pkls[f'Motion Template: {p.stem} ({p.name})'] = str(p)
    return src_imgs, drv_vids, drv_pkls

BUILTIN_SOURCES, BUILTIN_DRIVING_VIDS, BUILTIN_DRIVING_PKLS = list_builtin_examples()

class LivePortraitEngine:
    """High-level Colab engine around KwaiVGI/LivePortrait GradioPipeline."""
    def __init__(self):
        self.pipeline: Optional[GradioPipeline] = None

    def load(self, use_fp16: bool = True):
        if self.pipeline is not None:
            return
        print(f'[LivePortrait] Loading F, M, W, G, S networks & InsightFace/Landmark runners (fp16={use_fp16}) ...')
        t0 = time.time()
        args = ArgumentConfig(
            flag_use_half_precision=bool(use_fp16),
            output_dir=str(OUT_DIR),
        )
        inf_cfg = InferenceConfig(
            flag_use_half_precision=bool(use_fp16),
            output_fps=25,
        )
        crop_cfg = CropConfig()
        self.pipeline = GradioPipeline(inference_cfg=inf_cfg, crop_cfg=crop_cfg, args=args)
        print(f'[LivePortrait] Pipeline ready in {time.time() - t0:.1f}s | VRAM: {torch.cuda.memory_allocated() / (1024**3):.2f} GB')

    def animate_or_reenact(
        self,
        source_path: str,
        driving_path: str,
        is_source_video: bool = False,
        animation_region: str = 'all',
        driving_option: str = 'expression-friendly',
        driving_multiplier: float = 1.0,
        flag_relative_motion: bool = True,
        flag_stitching: bool = True,
        flag_pasteback: bool = True,
        flag_do_crop: bool = True,
        flag_crop_driving_video: bool = True,
        flag_normalize_lip: bool = False,
        scale: float = 2.3,
        vx_ratio: float = 0.0,
        vy_ratio: float = -0.125,
        driving_smooth_variance: float = 3e-7,
    ) -> Tuple[str, str]:
        self.load()
        if not source_path or not os.path.exists(str(source_path)):
            raise FileNotFoundError(f'Source file not found: {source_path}')
        if not driving_path or not os.path.exists(str(driving_path)):
            raise FileNotFoundError(f'Driving file not found: {driving_path}')

        user_args = {
            'source': str(source_path),
            'driving': str(driving_path),
            'output_dir': str(OUT_DIR),
            'flag_normalize_lip': bool(flag_normalize_lip),
            'flag_relative_motion': bool(flag_relative_motion),
            'flag_do_crop': bool(flag_do_crop),
            'flag_pasteback': bool(flag_pasteback),
            'flag_stitching': bool(flag_stitching),
            'animation_region': str(animation_region),
            'driving_option': str(driving_option),
            'driving_multiplier': float(driving_multiplier),
            'flag_crop_driving_video': bool(flag_crop_driving_video),
            'scale': float(scale),
            'vx_ratio': float(vx_ratio),
            'vy_ratio': float(vy_ratio),
            'driving_smooth_observation_variance': float(driving_smooth_variance),
        }
        self.pipeline.args = update_args(self.pipeline.args, user_args)
        self.pipeline.live_portrait_wrapper.update_config(self.pipeline.args.__dict__)
        self.pipeline.cropper.update_config(self.pipeline.args.__dict__)

        out_path, out_concat_path = self.pipeline.execute(self.pipeline.args)
        ext = pathlib.Path(out_path).suffix
        prefix = 'lp_v2v' if is_source_video else 'lp_i2v'
        stamp = f'{int(time.time())}_{uuid.uuid4().hex[:5]}'
        final_main = DRIVE_OUT / f'{prefix}_{stamp}{ext}'
        final_concat = DRIVE_OUT / f'{prefix}_{stamp}_concat{ext}'
        shutil.copy2(out_path, str(final_main))
        shutil.copy2(out_concat_path, str(final_concat))
        return str(final_main), str(final_concat)

    def retarget_video_lips(
        self,
        source_video_path: str,
        target_lip_ratio: float = 0.0,
        silence_lips: bool = True,
        source_scale: float = 2.3,
        smooth_variance: float = 1e-6,
        flag_do_crop: bool = True,
    ) -> Tuple[str, str]:
        self.load()
        out_concat, _, out_main, _ = self.pipeline.execute_video_retargeting(
            input_lip_ratio=float(target_lip_ratio),
            input_video=str(source_video_path),
            retargeting_source_scale=float(source_scale),
            driving_smooth_observation_variance_retargeting=float(smooth_variance),
            video_retargeting_silence=bool(silence_lips),
            flag_do_crop_input_retargeting_video=bool(flag_do_crop),
        )
        stamp = f'{int(time.time())}_{uuid.uuid4().hex[:5]}'
        final_main = DRIVE_OUT / f'lp_lipsilence_{stamp}.mp4'
        final_concat = DRIVE_OUT / f'lp_lipsilence_{stamp}_concat.mp4'
        shutil.copy2(out_main, str(final_main))
        shutil.copy2(out_concat, str(final_concat))
        return str(final_main), str(final_concat)

    def inspect_portrait_ratios(self, image_path: str, source_scale: float = 2.5) -> Tuple[float, float]:
        self.load()
        eye_r, lip_r = self.pipeline.init_retargeting_image(
            retargeting_source_scale=float(source_scale),
            source_eye_ratio=0.35,
            source_lip_ratio=0.05,
            input_image=str(image_path),
        )
        return float(eye_r), float(lip_r)

    def edit_portrait_expression(
        self,
        image_path: str,
        pitch: float = 0.0,
        yaw: float = 0.0,
        roll: float = 0.0,
        eye_ratio: Optional[float] = None,
        lip_ratio: Optional[float] = None,
        smile: float = 0.0,
        wink: float = 0.0,
        eyebrow: float = 0.0,
        eyeball_x: float = 0.0,
        eyeball_y: float = 0.0,
        lip_pout: float = 0.0,
        lip_purse: float = 0.0,
        lip_grin: float = 0.0,
        lip_open: float = 0.0,
        mov_x: float = 0.0,
        mov_y: float = 0.0,
        mov_z: float = 1.0,
        source_scale: float = 2.5,
        flag_stitching: bool = True,
        flag_do_crop: bool = True,
    ) -> Tuple[Image.Image, Image.Image, str]:
        self.load()
        det_eye, det_lip = self.inspect_portrait_ratios(image_path, source_scale=source_scale)
        use_eye = det_eye if eye_ratio is None else float(eye_ratio)
        use_lip = det_lip if lip_ratio is None else float(lip_ratio)
        out_crop_np, out_paste_np = self.pipeline.execute_image_retargeting(
            input_eye_ratio=use_eye,
            input_lip_ratio=use_lip,
            input_head_pitch_variation=float(pitch),
            input_head_yaw_variation=float(yaw),
            input_head_roll_variation=float(roll),
            mov_x=float(mov_x),
            mov_y=float(mov_y),
            mov_z=float(mov_z),
            lip_variation_zero=float(lip_pout),
            lip_variation_one=float(lip_purse),
            lip_variation_two=float(lip_grin),
            lip_variation_three=float(lip_open),
            smile=float(smile),
            wink=float(wink),
            eyebrow=float(eyebrow),
            eyeball_direction_x=float(eyeball_x),
            eyeball_direction_y=float(eyeball_y),
            input_image=str(image_path),
            retargeting_source_scale=float(source_scale),
            flag_stitching_retargeting_input=bool(flag_stitching),
            flag_do_crop_input_retargeting_image=bool(flag_do_crop),
        )
        pil_crop = Image.fromarray(out_crop_np)
        pil_paste = Image.fromarray(out_paste_np)
        out_png = DRIVE_OUT / f'lp_edited_portrait_{int(time.time())}_{uuid.uuid4().hex[:5]}.png'
        pil_paste.save(str(out_png))
        return pil_crop, pil_paste, str(out_png)

ENGINE = LivePortraitEngine()
print(f'[Core] LivePortraitEngine ready. Found {len(BUILTIN_SOURCES)} sample portraits, {len(BUILTIN_DRIVING_VIDS)} driving videos, and {len(BUILTIN_DRIVING_PKLS)} .pkl motion templates.')

In [ ]:
# @title STEP 4 — Launch Gradio UI (I2V Portrait Animator, V2V Reenactment & Interactive Expression Editor)
# @markdown Launches the 3-workflow Gradio Studio:
# @markdown 1. **🖼️ Image-to-Video Portrait Animation** (animate any portrait from a driving video or `.pkl` template)
# @markdown 2. **🎞️ Video-to-Video Facial Reenactment & Lip Silencing** (transfer expressions onto a video or silence lips)
# @markdown 3. **🎛️ Interactive Portrait Expression Editor** (Pitch/Yaw/Roll/Smile/Wink/Eyebrow/Mouth sliders for avatar prep)

import os, sys, time, shutil, pathlib, traceback
import torch, gradio as gr
from PIL import Image
from IPython.display import clear_output, display, FileLink

SRC_CHOICES = ['(Use Uploaded Portrait Below)'] + list(BUILTIN_SOURCES.keys())
DRV_VID_CHOICES = ['(Use Uploaded Driving Video Below)'] + list(BUILTIN_DRIVING_VIDS.keys())
DRV_PKL_CHOICES = ['(None — Use Driving Video)'] + list(BUILTIN_DRIVING_PKLS.keys())

def ui_run_i2v(
    src_preset, src_upload, drv_pkl_preset, drv_vid_preset, drv_upload,
    anim_region, drv_option, drv_mult, rel_motion, stitching, pasteback, do_crop, crop_drv, norm_lip,
    scale, vx, vy,
    progress=gr.Progress(),
):
    src_path = src_upload or BUILTIN_SOURCES.get(src_preset)
    if not src_path or not os.path.exists(str(src_path)):
        return None, None, '❌ Please upload a source portrait or select a built-in example portrait.'

    drv_path = None
    if drv_pkl_preset and not drv_pkl_preset.startswith('(None'):
        drv_path = BUILTIN_DRIVING_PKLS.get(drv_pkl_preset)
    if not drv_path:
        drv_path = drv_upload or BUILTIN_DRIVING_VIDS.get(drv_vid_preset)
    if not drv_path or not os.path.exists(str(drv_path)):
        return None, None, '❌ Please upload a driving video or select a built-in driving video / .pkl template.'

    try:
        progress(0.10, desc='Running LivePortrait Image-to-Video Animation ...')
        t0 = time.time()
        out_main, out_concat = ENGINE.animate_or_reenact(
            source_path=src_path,
            driving_path=drv_path,
            is_source_video=False,
            animation_region=anim_region,
            driving_option=drv_option,
            driving_multiplier=float(drv_mult),
            flag_relative_motion=bool(rel_motion),
            flag_stitching=bool(stitching),
            flag_pasteback=bool(pasteback),
            flag_do_crop=bool(do_crop),
            flag_crop_driving_video=bool(crop_drv),
            flag_normalize_lip=bool(norm_lip),
            scale=float(scale),
            vx_ratio=float(vx),
            vy_ratio=float(vy),
        )
        progress(1.0, desc='Done!')
        return out_main, out_concat, f'✅ **Animated in {time.time() - t0:.1f}s** → `{out_main}`'
    except Exception as exc:
        traceback.print_exc()
        return None, None, f'❌ Error: {exc}'

def ui_run_v2v(
    src_video, v2v_task, drv_pkl_preset, drv_vid_preset, drv_upload,
    anim_region, drv_option, drv_mult, smooth_var, target_lip_ratio, scale,
    progress=gr.Progress(),
):
    if not src_video or not os.path.exists(str(src_video)):
        return None, None, '❌ Please upload a source portrait video (.mp4).'
    try:
        t0 = time.time()
        if v2v_task.startswith('Lip-Silence'):
            progress(0.15, desc='Running V2V Lip-Silencing / Lip-Ratio Retargeting ...')
            out_main, out_concat = ENGINE.retarget_video_lips(
                source_video_path=src_video,
                target_lip_ratio=float(target_lip_ratio),
                silence_lips=(float(target_lip_ratio) == 0.0),
                source_scale=float(scale),
                smooth_variance=float(smooth_var),
            )
            return out_main, out_concat, f'✅ **V2V Lip Retargeting completed in {time.time() - t0:.1f}s** → `{out_main}`'

        drv_path = None
        if drv_pkl_preset and not drv_pkl_preset.startswith('(None'):
            drv_path = BUILTIN_DRIVING_PKLS.get(drv_pkl_preset)
        if not drv_path:
            drv_path = drv_upload or BUILTIN_DRIVING_VIDS.get(drv_vid_preset)
        if not drv_path or not os.path.exists(str(drv_path)):
            return None, None, '❌ Please choose a driving video or .pkl template for V2V reenactment.'

        progress(0.15, desc='Running V2V Facial Reenactment ...')
        out_main, out_concat = ENGINE.animate_or_reenact(
            source_path=src_video,
            driving_path=drv_path,
            is_source_video=True,
            animation_region=anim_region,
            driving_option=drv_option,
            driving_multiplier=float(drv_mult),
            scale=float(scale),
            driving_smooth_variance=float(smooth_var),
        )
        return out_main, out_concat, f'✅ **V2V Facial Reenactment completed in {time.time() - t0:.1f}s** → `{out_main}`'
    except Exception as exc:
        traceback.print_exc()
        return None, None, f'❌ Error: {exc}'

def on_edit_image_upload(img_path, src_preset, scale):
    use_path = img_path or BUILTIN_SOURCES.get(src_preset)
    if not use_path or not os.path.exists(str(use_path)):
        return gr.update(), gr.update(), 'Upload a portrait to detect initial eye & lip ratios.'
    try:
        eye_r, lip_r = ENGINE.inspect_portrait_ratios(use_path, source_scale=float(scale))
        return gr.update(value=eye_r), gr.update(value=lip_r), f'🔍 Detected initial **Eye-Open Ratio = `{eye_r}`**, **Lip-Open Ratio = `{lip_r}`**.'
    except Exception as exc:
        return gr.update(), gr.update(), f'⚠️ Could not detect face landmarks: {exc}'

def ui_run_editor(
    src_preset, img_upload, pitch, yaw, roll, eye_ratio, lip_ratio,
    smile, wink, eyebrow, eye_x, eye_y, lip_pout, lip_purse, lip_grin, lip_open,
    mov_x, mov_y, mov_z, scale, stitching, do_crop,
):
    use_path = img_upload or BUILTIN_SOURCES.get(src_preset)
    if not use_path or not os.path.exists(str(use_path)):
        return None, None, '❌ Please upload a portrait or select a built-in example portrait.'
    try:
        t0 = time.time()
        pil_crop, pil_paste, out_png = ENGINE.edit_portrait_expression(
            image_path=use_path,
            pitch=float(pitch),
            yaw=float(yaw),
            roll=float(roll),
            eye_ratio=float(eye_ratio),
            lip_ratio=float(lip_ratio),
            smile=float(smile),
            wink=float(wink),
            eyebrow=float(eyebrow),
            eyeball_x=float(eye_x),
            eyeball_y=float(eye_y),
            lip_pout=float(lip_pout),
            lip_purse=float(lip_purse),
            lip_grin=float(lip_grin),
            lip_open=float(lip_open),
            mov_x=float(mov_x),
            mov_y=float(mov_y),
            mov_z=float(mov_z),
            source_scale=float(scale),
            flag_stitching=bool(stitching),
            flag_do_crop=bool(do_crop),
        )
        return pil_paste, pil_crop, f'✅ **Edited portrait in {(time.time() - t0)*1000:.0f} ms** → Saved full-res PNG to `{out_png}`'
    except Exception as exc:
        traceback.print_exc()
        return None, None, f'❌ Error: {exc}'

with gr.Blocks(title='LivePortrait Studio', theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        '# 🎭 LivePortrait — Portrait Animation, V2V Facial Reenactment & Interactive Expression Editor\n'
        'Animate static portraits, transfer expressions onto videos, or fine-tune head pose & mouth closure before feeding portraits into `InfiniteTalk` / `LongCat-Avatar`.'
    )

    with gr.Tabs():
        with gr.Tab('1. 🖼️ Image-to-Video Portrait Animation (I2V)'):
            with gr.Row():
                with gr.Column(scale=3):
                    with gr.Row():
                        i2v_src_dd = gr.Dropdown(
                            choices=SRC_CHOICES, value=SRC_CHOICES[1] if len(SRC_CHOICES) > 1 else SRC_CHOICES[0],
                            label='Built-In Example Portrait',
                            info='Pick a sample portrait or upload your own portrait image below.',
                        )
                        i2v_pkl_dd = gr.Dropdown(
                            choices=DRV_PKL_CHOICES, value=DRV_PKL_CHOICES[0],
                            label='Built-In Motion Template (.pkl)',
                            info='Select a lightweight motion template (talking, laugh, wink, shy) or leave on None to use a driving video.',
                        )
                        i2v_drv_dd = gr.Dropdown(
                            choices=DRV_VID_CHOICES, value=DRV_VID_CHOICES[1] if len(DRV_VID_CHOICES) > 1 else DRV_VID_CHOICES[0],
                            label='Built-In Driving Video (.mp4)',
                            info='Used when no uploaded driving video or .pkl template is selected.',
                        )
                    with gr.Row():
                        i2v_src_img = gr.Image(type='filepath', label='Upload Custom Source Portrait (.png / .jpg)', height=240)
                        i2v_drv_vid = gr.Video(label='Upload Custom Driving Facial Video (.mp4)', height=240)
                    with gr.Row():
                        i2v_region = gr.Dropdown(
                            choices=['all', 'exp', 'pose', 'lip', 'eyes'], value='all',
                            label='Animation Region',
                            info="'all' = pose + expression; 'exp' = expression only (locks head angle); 'lip' = mouth only; 'eyes' = blink/gaze only.",
                        )
                        i2v_drv_opt = gr.Dropdown(
                            choices=['expression-friendly', 'pose-friendly'], value='expression-friendly',
                            label='Driving Motion Mode',
                            info="'expression-friendly' preserves subtle facial expressions accurately; 'pose-friendly' handles large head rotations.",
                        )
                        i2v_mult = gr.Slider(0.2, 2.0, value=1.0, step=0.05, label='Driving Multiplier', info='Scales the amplitude of transferred facial motion (default 1.0).')
                    with gr.Accordion('⚙️ Stitching, Lip Normalization & Face Crop Options', open=False):
                        with gr.Row():
                            i2v_rel = gr.Checkbox(value=True, label='Relative Motion', info='Computes motion deltas relative to frame 0 of the driving video.')
                            i2v_stitch = gr.Checkbox(value=True, label='Enable Stitching Module', info='Seamlessly blends the animated face back into the original neck/shoulder boundary.')
                            i2v_paste = gr.Checkbox(value=True, label='Paste-Back to Full Image', info='Outputs the full uncropped portrait frame rather than just the 512x512 face crop.')
                            i2v_norm_lip = gr.Checkbox(value=False, label='Normalize Source Lips (Close Mouth First)', info='Neutralizes open lips on the source portrait before applying driving motion.')
                        with gr.Row():
                            i2v_crop_src = gr.Checkbox(value=True, label='Auto-Crop Source Portrait', info='Detects and aligns the face region automatically.')
                            i2v_crop_drv = gr.Checkbox(value=True, label='Auto-Crop Driving Video', info='Tracks and crops the face in non-square driving videos.')
                            i2v_scale = gr.Slider(1.8, 3.2, value=2.3, step=0.1, label='Source Crop Scale', info='Larger values include more hair/shoulders around the face crop.')
                            i2v_vx = gr.Slider(-0.3, 0.3, value=0.0, step=0.025, label='Crop Horizontal Offset (vx)', info='Horizontal face crop shift.')
                            i2v_vy = gr.Slider(-0.3, 0.3, value=-0.125, step=0.025, label='Crop Vertical Offset (vy)', info='Vertical face crop shift (default -0.125).')
                    i2v_btn = gr.Button('🎬 Animate Portrait (I2V)', variant='primary')
                with gr.Column(scale=2):
                    i2v_out_main = gr.Video(label='Animated Portrait Video', height=300)
                    i2v_out_concat = gr.Video(label='Side-by-Side Comparison (Driving | Source | Animated)', height=220)
                    i2v_status = gr.Markdown('Ready.')
            i2v_btn.click(
                ui_run_i2v,
                inputs=[
                    i2v_src_dd, i2v_src_img, i2v_pkl_dd, i2v_drv_dd, i2v_drv_vid,
                    i2v_region, i2v_drv_opt, i2v_mult, i2v_rel, i2v_stitch, i2v_paste, i2v_crop_src, i2v_crop_drv, i2v_norm_lip,
                    i2v_scale, i2v_vx, i2v_vy,
                ],
                outputs=[i2v_out_main, i2v_out_concat, i2v_status],
            )

        with gr.Tab('2. 🎞️ Video-to-Video Facial Reenactment & Lip Silencing (V2V)'):
            with gr.Row():
                with gr.Column(scale=3):
                    v2v_task_dd = gr.Dropdown(
                        choices=[
                            'Facial Reenactment (Transfer Driving Expression/Pose onto Source Video)',
                            'Lip-Silence / Lip-Ratio Retargeting (Close Mouth Across Source Video for Idle Loops)',
                        ],
                        value='Facial Reenactment (Transfer Driving Expression/Pose onto Source Video)',
                        label='V2V Editing Mode',
                        info='Choose full V2V facial reenactment or V2V lip-silencing (ideal for creating silent listening loops).',
                    )
                    with gr.Row():
                        v2v_src_vid = gr.Video(label='Source Portrait Video (.mp4)', height=240)
                        v2v_drv_vid = gr.Video(label='Driving Facial Video (.mp4 — for Reenactment)', height=240)
                    with gr.Row():
                        v2v_pkl_dd = gr.Dropdown(choices=DRV_PKL_CHOICES, value=DRV_PKL_CHOICES[0], label='Or Built-In Driving Template (.pkl)', info='Optional .pkl driving template.')
                        v2v_drv_dd = gr.Dropdown(choices=DRV_VID_CHOICES, value=DRV_VID_CHOICES[1] if len(DRV_VID_CHOICES) > 1 else DRV_VID_CHOICES[0], label='Or Built-In Driving Video', info='Fallback built-in driving clip.')
                    with gr.Row():
                        v2v_region = gr.Dropdown(choices=['all', 'exp', 'pose', 'lip', 'eyes'], value='exp', label='V2V Target Region', info="'exp' transfers expressions/lips while keeping the source video's original head trajectory.")
                        v2v_drv_opt = gr.Dropdown(choices=['expression-friendly', 'pose-friendly'], value='pose-friendly', label='Driving Mode', info='pose-friendly is recommended for V2V editing.')
                        v2v_mult = gr.Slider(0.2, 2.0, value=1.0, step=0.05, label='Driving Multiplier', info='Strength of transferred expression.')
                    with gr.Row():
                        v2v_smooth = gr.Slider(1e-7, 1e-4, value=3e-7, step=1e-7, label='Kalman Temporal Smoothing Variance', info='Higher values smooth inter-frame motion jitter on source videos.')
                        v2v_lip_ratio = gr.Slider(0.0, 0.8, value=0.0, step=0.02, label='Target Lip-Open Ratio (0.0 = Fully Closed / Silent)', info='Used in Lip-Silence mode to close or open the speaker mouth across all video frames.')
                        v2v_scale = gr.Slider(1.8, 3.2, value=2.3, step=0.1, label='Source Video Crop Scale', info='Face crop scale on source video.')
                    v2v_btn = gr.Button('🎞️ Run V2V Reenactment / Lip Retargeting', variant='primary')
                with gr.Column(scale=2):
                    v2v_out_main = gr.Video(label='Edited V2V Output Video', height=300)
                    v2v_out_concat = gr.Video(label='Side-by-Side Comparison', height=220)
                    v2v_status = gr.Markdown('Ready.')
            v2v_btn.click(
                ui_run_v2v,
                inputs=[
                    v2v_src_vid, v2v_task_dd, v2v_pkl_dd, v2v_drv_dd, v2v_drv_vid,
                    v2v_region, v2v_drv_opt, v2v_mult, v2v_smooth, v2v_lip_ratio, v2v_scale,
                ],
                outputs=[v2v_out_main, v2v_out_concat, v2v_status],
            )

        with gr.Tab('3. 🎛️ Interactive Portrait Expression Editor (Pitch / Yaw / Roll / Mouth / Smile)'):
            gr.Markdown(
                '**Avatar Portrait Prep Tool:** Upload any portrait (e.g., Nikita avatar keyframes) to gently close an open mouth (`Lip-Open Ratio = 0.0` / `Lip Open/Close`), '
                'adjust head angle (`Pitch`, `Yaw`, `Roll`), or add a subtle smile/wink before passing the image into `InfiniteTalk` or `LongCat-Video-Avatar`.'
            )
            with gr.Row():
                with gr.Column(scale=3):
                    with gr.Row():
                        ed_src_dd = gr.Dropdown(choices=SRC_CHOICES, value=SRC_CHOICES[1] if len(SRC_CHOICES) > 1 else SRC_CHOICES[0], label='Built-In Example Portrait', info='Select a sample portrait or upload your own below.')
                        ed_scale = gr.Slider(1.8, 3.2, value=2.5, step=0.1, label='Face Crop Scale', info='2.5 is the upstream default for single-image retargeting.')
                        ed_stitch = gr.Checkbox(value=True, label='Enable Stitching', info='Keeps neck/hair boundary locked while editing expression.')
                        ed_crop = gr.Checkbox(value=True, label='Paste-Back to Full Image', info='Pastes the edited face seamlessly back onto the full-resolution image.')
                    ed_img_in = gr.Image(type='filepath', label='Upload Portrait Image to Edit (.png / .jpg)', height=220)
                    with gr.Row():
                        ed_pitch = gr.Slider(-15.0, 15.0, value=0.0, step=0.5, label='Head Pitch (Up / Down °)', info='Relative vertical head tilt.')
                        ed_yaw = gr.Slider(-25.0, 25.0, value=0.0, step=0.5, label='Head Yaw (Left / Right °)', info='Relative horizontal head turn.')
                        ed_roll = gr.Slider(-15.0, 15.0, value=0.0, step=0.5, label='Head Roll (Tilt °)', info='Relative head roll angle.')
                    with gr.Row():
                        ed_eye_ratio = gr.Slider(0.0, 0.8, value=0.35, step=0.01, label='Target Eye-Open Ratio', info='0.0 = closed eyes; 0.35–0.45 = natural open eyes.')
                        ed_lip_ratio = gr.Slider(0.0, 0.8, value=0.0, step=0.01, label='Target Lip-Open Ratio (0.0 = Gently Close Mouth)', info='Set to 0.0 to close an open mouth cleanly.')
                        ed_smile = gr.Slider(-0.3, 1.0, value=0.0, step=0.05, label='Smile', info='Positive = warm smile; negative = serious/frown.')
                        ed_wink = gr.Slider(0.0, 39.0, value=0.0, step=1.0, label='Wink', info='Winks one eye.')
                    with gr.Row():
                        ed_eyebrow = gr.Slider(-30.0, 30.0, value=0.0, step=1.0, label='Eyebrow Raise / Furrow', info='Adjusts eyebrow height.')
                        ed_eye_x = gr.Slider(-30.0, 30.0, value=0.0, step=1.0, label='Eyeball Gaze Horizontal (X)', info='Shifts pupil gaze left/right.')
                        ed_eye_y = gr.Slider(-63.0, 63.0, value=0.0, step=1.0, label='Eyeball Gaze Vertical (Y)', info='Shifts pupil gaze up/down.')
                    with gr.Accordion('👄 Fine Mouth & Translation Micro-Sliders', open=False):
                        with gr.Row():
                            ed_lip_pout = gr.Slider(-0.09, 0.09, value=0.0, step=0.005, label='Lip Pouting (v0)', info='Forward lip pout.')
                            ed_lip_purse = gr.Slider(-20.0, 15.0, value=0.0, step=0.5, label='Lip Pursing (v1)', info='Narrow/purse lips.')
                            ed_lip_grin = gr.Slider(0.0, 15.0, value=0.0, step=0.5, label='Lip Grin (v2)', info='Wide mouth grin.')
                            ed_lip_open = gr.Slider(-90.0, 120.0, value=0.0, step=2.0, label='Jaw / Mouth Open-Close (v3)', info='Negative values pull the jaw shut; positive values open the mouth.')
                        with gr.Row():
                            ed_mov_x = gr.Slider(-0.1, 0.1, value=0.0, step=0.005, label='Head Shift X', info='Micro horizontal translation.')
                            ed_mov_y = gr.Slider(-0.1, 0.1, value=0.0, step=0.005, label='Head Shift Y', info='Micro vertical translation.')
                            ed_mov_z = gr.Slider(0.9, 1.1, value=1.0, step=0.01, label='Head Scale Z', info='Micro zoom factor (1.0 = neutral).')
                    ed_btn = gr.Button('✨ Apply Expression Edit & Save High-Res PNG', variant='primary')
                with gr.Column(scale=2):
                    ed_out_paste = gr.Image(label='Edited Full-Resolution Portrait (Saved to Drive)', height=320)
                    ed_out_crop = gr.Image(label='Edited 512×512 Face Crop', height=220)
                    ed_status = gr.Markdown('Upload or select a portrait and click **Apply Expression Edit**.')

            ed_img_in.change(on_edit_image_upload, inputs=[ed_img_in, ed_src_dd, ed_scale], outputs=[ed_eye_ratio, ed_lip_ratio, ed_status])
            ed_src_dd.change(on_edit_image_upload, inputs=[ed_img_in, ed_src_dd, ed_scale], outputs=[ed_eye_ratio, ed_lip_ratio, ed_status])
            ed_btn.click(
                ui_run_editor,
                inputs=[
                    ed_src_dd, ed_img_in, ed_pitch, ed_yaw, ed_roll, ed_eye_ratio, ed_lip_ratio,
                    ed_smile, ed_wink, ed_eyebrow, ed_eye_x, ed_eye_y, ed_lip_pout, ed_lip_purse, ed_lip_grin, ed_lip_open,
                    ed_mov_x, ed_mov_y, ed_mov_z, ed_scale, ed_stitch, ed_crop,
                ],
                outputs=[ed_out_paste, ed_out_crop, ed_status],
            )

clear_output()
print('[UI] Launching LivePortrait Gradio Studio ...')
demo.queue(max_size=4, concurrency_limit=1)
demo.load(lambda: '✅ **LivePortrait Studio ready** — choose I2V Portrait Animation, V2V Reenactment, or Interactive Expression Editor.', outputs=[i2v_status])
demo.launch(share=True, show_error=True, height=900, prevent_thread_lock=False)

In [ ]:
# @title STEP 5 — Keep-Alive (anti-disconnect)
# @markdown Pings Hugging Face API every 60 seconds in a background thread to keep the Colab session active.

import time, threading, datetime, requests, IPython
_stop_keepalive = threading.Event()

def _pinger():
    while not _stop_keepalive.is_set():
        try:
            requests.get('https://huggingface.co/api/models/KwaiVGI/LivePortrait', timeout=5)
            IPython.display.clear_output(wait=True)
            print(f"[Keep-Alive] {datetime.datetime.now().strftime('%H:%M:%S')} OK — stop with `_stop_keepalive.set()`.")
        except Exception as exc:
            print(f'[Keep-Alive] Warning: {exc}')
        _stop_keepalive.wait(60)

_ka_thread = threading.Thread(target=_pinger, daemon=True)
_ka_thread.start()
print('[Keep-Alive] Daemon started.')

In [ ]:
# @title STEP 6 — Quick Test (Headless Portrait Expression Edit + I2V Animation)
# @markdown Verifies both single-image expression editing and I2V portrait animation on a sample portrait.

import os, time, traceback
from IPython.display import display, FileLink

try:
    sample_src = str(SOURCE_EXAMPLES_DIR / 's0.jpg')
    sample_drv = str(DRIVING_EXAMPLES_DIR / 'talking.pkl') if (DRIVING_EXAMPLES_DIR / 'talking.pkl').exists() else str(DRIVING_EXAMPLES_DIR / 'd0.mp4')
    t0 = time.time()
    _, _, edited_png = ENGINE.edit_portrait_expression(
        image_path=sample_src,
        pitch=-3.0,
        yaw=4.0,
        smile=0.35,
        lip_ratio=0.0,
    )
    print(f'[QuickTest 1/2] Portrait expression edit completed in {time.time() - t0:.2f}s -> {edited_png}')
    display(FileLink(str(edited_png)))

    t1 = time.time()
    anim_mp4, _ = ENGINE.animate_or_reenact(source_path=edited_png, driving_path=sample_drv)
    print(f'[QuickTest 2/2] Portrait I2V animation completed in {time.time() - t1:.2f}s -> {anim_mp4}')
    display(FileLink(str(anim_mp4)))
except Exception as exc:
    traceback.print_exc()
    raise

In [ ]:
# @title STEP 7 — Batch Portrait Animation from Folder
# @markdown Animates a list/folder of source portraits against a chosen driving video or `.pkl` motion template
# @markdown with per-item `try`/`except` error handling.

import os, time, pathlib, traceback
from tqdm import tqdm
from IPython.display import display, FileLink

BATCH_SOURCE_DIR = str(SOURCE_EXAMPLES_DIR)  # @param {type:"string"}
BATCH_DRIVING_PATH = str(DRIVING_EXAMPLES_DIR / 'talking.pkl')  # @param {type:"string"}
BATCH_MAX_PORTRAITS = 3  # @param {type:"integer"}

src_dir = pathlib.Path(BATCH_SOURCE_DIR)
portraits = sorted([p for p in src_dir.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png', '.webp')])[:BATCH_MAX_PORTRAITS]
print(f'[Batch] Animating {len(portraits)} portrait(s) with driving motion {BATCH_DRIVING_PATH} -> {DRIVE_OUT}')

ok_count, fail_count = 0, 0
for idx, p_img in enumerate(tqdm(portraits, desc='LivePortrait Batch'), 1):
    try:
        t0 = time.time()
        out_mp4, _ = ENGINE.animate_or_reenact(
            source_path=str(p_img),
            driving_path=BATCH_DRIVING_PATH,
        )
        print(f'  [{idx:02d}/{len(portraits)}] OK ({time.time() - t0:.1f}s) -> {out_mp4}')
        display(FileLink(str(out_mp4)))
        ok_count += 1
    except Exception as exc:
        fail_count += 1
        print(f'  [{idx:02d}/{len(portraits)}] FAILED on {p_img.name}: {exc}')
        traceback.print_exc()

print(f'[Batch] Finished: {ok_count} succeeded, {fail_count} failed. Output folder: {DRIVE_OUT}')